In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


In [1]:
from pyspark.sql import functions as F

marketing = spark.table("lh_silver_game.marketing_spend_clean")

StatementMeta(, 9cdb1655-d195-438d-b7a1-e860b2204a6e, 3, Finished, Available, Finished, False)

In [2]:
marketing_performance = (
    marketing
    .groupBy("acquisition_channel")
    .agg(
        F.sum("impressions").alias("impressions"),
        F.sum("clicks").alias("clicks"),
        F.sum("installs").alias("installs"),
        F.sum("spend_usd").alias("spend_usd")
    )
    .withColumn(
        "ctr",
        F.col("clicks") / F.col("impressions")
    )
    .withColumn(
        "cvr",
        F.col("installs") / F.col("clicks")
    )
    .withColumn(
        "cpc",
        F.col("spend_usd") / F.col("clicks")
    )
    .withColumn(
        "cpi",
        F.col("spend_usd") / F.col("installs")
    )
)

StatementMeta(, 9cdb1655-d195-438d-b7a1-e860b2204a6e, 4, Finished, Available, Finished, False)

In [3]:
monetization = spark.table(
    "lh_gold_game.monetization_metrics"
)

marketing_performance = (
    marketing_performance
    .join(
        monetization.select(
            "acquisition_channel",
            "payers",
            "total_revenue",
            "payer_conversion",
            "arpu",
            "arppu"
        ),
        on="acquisition_channel",
        how="left"
    )
)

StatementMeta(, 9cdb1655-d195-438d-b7a1-e860b2204a6e, 5, Finished, Available, Finished, False)

In [4]:
marketing_performance = (
    marketing_performance
    .withColumn(
        "cac",
        F.when(
            F.col("payers") > 0,
            F.col("spend_usd") / F.col("payers")
        )
    )
    .withColumn(
        "roas",
        F.when(
            F.col("spend_usd") > 0,
            F.col("total_revenue") / F.col("spend_usd")
        )
    )
)

StatementMeta(, 9cdb1655-d195-438d-b7a1-e860b2204a6e, 6, Finished, Available, Finished, False)

In [5]:
display(
    marketing_performance.orderBy("acquisition_channel")
)

StatementMeta(, 9cdb1655-d195-438d-b7a1-e860b2204a6e, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e9b912c5-c042-490e-bc47-c1cb8aed3c73)

In [6]:
marketing_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("lh_gold_game.marketing_performance")

StatementMeta(, 9cdb1655-d195-438d-b7a1-e860b2204a6e, 8, Finished, Available, Finished, False)

In [7]:
df_check = spark.table("lh_gold_game.marketing_performance")

print("Saved row count:", df_check.count())
display(df_check)

StatementMeta(, 9cdb1655-d195-438d-b7a1-e860b2204a6e, 9, Finished, Available, Finished, False)

Saved row count: 4


SynapseWidget(Synapse.DataFrame, e45bace6-f8d9-40b3-8529-99b555cef623)